In [25]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

In [26]:
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw4_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [27]:
import kagglehub

dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(dataset_dir)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
/kaggle/input/brazilian-ecommerce


## 1. Setup and Olist data loading

У цій частині я запускаю PostgreSQL через pgserver та завантажую Olist Brazilian E-commerce Dataset. Датасет містить декілька пов'язаних таблиць, які далі будуть використані для створення raw та typed-схеми.

### Raw tables

Спочатку завантажую вихідні CSV-файли в raw-таблиці. На цьому етапі дані зберігаються у початковому вигляді, а typed-таблиці з правильними типами даних і constraints будуть створені окремо.

In [28]:
import os

files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
    'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(
        table_name,
        engine,
        if_exists='replace',
        index=False,
        chunksize=10_000
    )
    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)
olist_order_reviews_raw (99224, 7)


### Typed schema

Тепер створюю typed-таблиці з правильними типами даних, ключами та обмеженнями. Саме ці таблиці буду використовувати у наступних SQL-запитах.

In [29]:
typed_schema_sql = """
DROP TABLE IF EXISTS customer_segments    CASCADE;
DROP TABLE IF EXISTS seller_score         CASCADE;
DROP TABLE IF EXISTS seller_alerts        CASCADE;
DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
DROP TABLE IF EXISTS olist_order_items    CASCADE;
DROP TABLE IF EXISTS olist_orders         CASCADE;
DROP TABLE IF EXISTS olist_customers      CASCADE;
DROP TABLE IF EXISTS olist_products       CASCADE;
DROP TABLE IF EXISTS olist_sellers        CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id
    ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order
    ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id                TEXT,
    order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title     TEXT,
    review_comment_message   TEXT,
    review_creation_date     TIMESTAMP,
    review_answer_timestamp  TIMESTAMP
);

CREATE INDEX idx_olist_order_reviews_order_id
    ON olist_order_reviews (order_id);
"""

with engine.begin() as conn:
    conn.execute(text(typed_schema_sql))

print("Typed tables created successfully")

Typed tables created successfully


In [30]:
load_typed_sql = """
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
SELECT
    review_id,
    order_id,
    review_score::SMALLINT,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;
"""

with engine.begin() as conn:
    conn.execute(text(load_typed_sql))

print("Data loaded into typed tables successfully")

Data loaded into typed tables successfully


### Перевірка кількості рядків

Перевіряю кількість рядків у typed-таблицях, щоб переконатися, що під час перенесення даних з raw-таблиць нічого не було втрачено.

In [31]:
row_count_sql = """
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL
SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL
SELECT 'products', COUNT(*) FROM olist_products
UNION ALL
SELECT 'sellers', COUNT(*) FROM olist_sellers
UNION ALL
SELECT 'reviews', COUNT(*) FROM olist_order_reviews
ORDER BY table_name;
"""

pd.read_sql(row_count_sql, engine)

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,reviews,99224
5,sellers,3095


Кількість рядків у typed-таблицях відповідає кількості рядків у raw-таблицях. Отже, під час перенесення даних рядки не були втрачені.

## 2. DML-запити

У цій частині використовую DML-команди PostgreSQL для додавання, оновлення та видалення даних. Також перевіряю роботу RETURNING та ON CONFLICT.

### 2.1 INSERT ... RETURNING

Створюю alert для продавця з найбільшою кількістю запізнілих доставок. RETURNING дозволяє одразу побачити створений запис без додаткового SELECT.

In [32]:
insert_alert_sql = """
DROP TABLE IF EXISTS seller_alerts CASCADE;

CREATE TABLE seller_alerts (
    alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL
        CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details    TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT
        so.seller_id,
        COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o
        ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE >
          o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (
    seller_id,
    alert_type,
    severity,
    details
)
SELECT
    seller_id,
    'late_delivery',
    4,
    'Seller has ' || late_orders ||
    ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id, seller_id, alert_type, severity, created_at;
"""

with engine.begin() as conn:
    result = conn.execute(text(insert_alert_sql))
    rows = result.fetchall()
    columns = result.keys()

pd.DataFrame(rows, columns=columns)

,alert_id,seller_id,alert_type,severity,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,2026-09-28 15:15:53.555417+00:00


Запит створив alert для продавця з найбільшою кількістю запізнілих доставок. RETURNING одразу повертає створений запис, тому результат можна перевірити без додаткового SELECT і використати для audit-log.

### 2.2 INSERT ... ON CONFLICT DO UPDATE

Створюю рейтинг продавців на основі відгуків. Спочатку дані приводжу до рівня seller_id × order_id, щоб один відгук не враховувався декілька разів через кілька товарів в одному замовленні.

In [33]:
seller_score_sql = """
DROP TABLE IF EXISTS seller_score CASCADE;

CREATE TABLE seller_score (
    seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating  NUMERIC(3, 2),
    n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier        TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
SELECT
    seller_id,
    avg_rating,
    n_reviews,
    CASE
        WHEN avg_rating IS NULL THEN NULL
        WHEN avg_rating >= 4.5 THEN 'gold'
        WHEN avg_rating >= 3.5 THEN 'silver'
        ELSE 'bronze'
    END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET avg_rating = EXCLUDED.avg_rating,
    n_reviews  = EXCLUDED.n_reviews,
    tier       = EXCLUDED.tier,
    updated_at = NOW()
RETURNING seller_id, avg_rating, n_reviews, tier, updated_at;
"""

with engine.begin() as conn:
    result = conn.execute(text(seller_score_sql))
    rows = result.fetchall()
    columns = result.keys()

seller_score_result = pd.DataFrame(rows, columns=columns)

print("Number of sellers:", len(seller_score_result))
seller_score_result.head(10)

Number of sellers: 3095


,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-09-28 15:15:53.922295+00:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-09-28 15:15:53.922295+00:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-09-28 15:15:53.922295+00:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-09-28 15:15:53.922295+00:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-09-28 15:15:53.922295+00:00
5,fc38b5dceee1a730fad8853453437fbd,4.60,5,gold,2026-09-28 15:15:53.922295+00:00
6,5def4c3732941a971cba8fdee992ede1,4.60,5,gold,2026-09-28 15:15:53.922295+00:00
7,b39d7fe263ef469605dbb32608aee0af,3.88,43,silver,2026-09-28 15:15:53.922295+00:00
8,ea65d8b58316a6f2362f2a9e4b3e86ad,4.00,9,silver,2026-09-28 15:15:53.922295+00:00
9,a45765f8afb1e594b22bf9e974b46765,4.00,1,silver,2026-09-28 15:15:53.922295+00:00


`SELECT DISTINCT` використовую, щоб одне замовлення продавця не повторювалося через декілька товарів. Відгуки спочатку агрегуються на рівні `order_id`, тому один review не рахується декілька разів.

`ON CONFLICT (seller_id) DO UPDATE` дозволяє повторно запускати запит без створення дублікатів - існуючі дані просто оновлюються.

Якщо у продавця немає відгуків, `avg_rating` і `tier` можуть бути `NULL`, а `n_reviews` дорівнює 0. Також review в Olist прив'язаний до замовлення, а не до конкретного продавця, тому для замовлень з декількома продавцями це є навчальним спрощенням.

### 2.3 UPDATE ... FROM

Додаю до замовлень ознаку `is_late`, яка показує, чи була доставка виконана пізніше запланованої дати. Запит можна запускати повторно, оскільки значення щоразу перераховується на основі актуальних дат доставки.

In [34]:
update_late_sql = """
ALTER TABLE olist_orders
    ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;

WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE >
                order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
RETURNING o.order_id, o.customer_id, o.is_late;
"""

with engine.begin() as conn:
    result = conn.execute(text(update_late_sql))
    rows = result.fetchall()
    columns = result.keys()

late_result = pd.DataFrame(rows, columns=columns)

print("Updated orders:", len(late_result))
print("Late orders:", late_result["is_late"].sum())

late_result.head(10)

Updated orders: 99441
Late orders: 6535


,order_id,customer_id,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,False
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,False
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,False
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,False
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,False
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,False


### 2.4 DELETE ... RETURNING

Видаляю alerts, які були створені більше 30 днів тому. RETURNING показує, які саме записи були видалені. Якщо таких записів немає, порожній результат є нормальним.

In [35]:
delete_alerts_sql = """
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;
"""

with engine.begin() as conn:
    result = conn.execute(text(delete_alerts_sql))
    rows = result.fetchall()
    columns = result.keys()

deleted_alerts = pd.DataFrame(rows, columns=columns)

print("Deleted alerts:", len(deleted_alerts))
deleted_alerts

Deleted alerts: 0


,alert_id,seller_id,alert_type,created_at


Видалених записів немає, оскільки в таблиці зараз немає alerts, створених більше 30 днів тому.

## 3. Customer segmentation

Сегментацію виконую на рівні `customer_unique_id`, оскільки він краще ідентифікує реального покупця. `customer_id` може бути різним для різних замовлень одного покупця.

Спочатку створюю `olist_customer_dim`, де кожен `customer_unique_id` представлений лише один раз.

In [36]:
customer_dim_sql = """
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);

INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id AS latest_customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp DESC NULLS LAST,
    c.customer_id;
"""

with engine.begin() as conn:
    conn.execute(text(customer_dim_sql))

customer_dim_check = pd.read_sql("""
SELECT COUNT(*) AS unique_customers
FROM olist_customer_dim;
""", engine)

customer_dim_check

,unique_customers
0,96096


### Таблиця customer_segments

Створюю таблицю для збереження сегментації покупців. Для `customer_unique_id` використовую UNIQUE та FOREIGN KEY, а для інших полів додаю constraints, щоб контролювати коректність даних.

In [37]:
customer_segments_table_sql = """
CREATE TABLE customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

    rfm_score           SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days        INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),

    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders            INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;
"""

with engine.begin() as conn:
    conn.execute(text(customer_segments_table_sql))

print("customer_segments table created successfully")

customer_segments table created successfully


### Заповнення customer_segments

Розраховую для кожного покупця кількість замовлень, загальну суму покупок та кількість днів від останнього замовлення. На основі цих показників визначаю сегмент покупця та RFM score.

In [38]:
customer_segments_insert_sql = """
WITH customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS n_orders,
        COALESCE(SUM(oi.price), 0)::NUMERIC(12, 2) AS monetary_value,
        MAX(o.order_purchase_timestamp) AS last_order_date
    FROM olist_customers AS c
    LEFT JOIN olist_orders AS o
        ON o.customer_id = c.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY c.customer_unique_id
),
rfm AS (
    SELECT
        customer_unique_id,
        n_orders,
        monetary_value,
        CASE
            WHEN last_order_date IS NULL THEN NULL
            ELSE (
                MAX(last_order_date) OVER ()::DATE
                - last_order_date::DATE
            )
        END AS recency_days
    FROM customer_stats
)
INSERT INTO customer_segments (
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
)
SELECT
    customer_unique_id,
    CASE
        WHEN n_orders >= 3 AND monetary_value >= 500 THEN 'VIP'
        WHEN recency_days <= 90 THEN 'new'
        WHEN recency_days > 365 THEN 'inactive'
        ELSE 'regular'
    END AS segment_name,
    CASE
        WHEN n_orders >= 3 AND monetary_value >= 500 THEN 5
        WHEN recency_days <= 90 THEN 4
        WHEN recency_days > 365 THEN 1
        ELSE 3
    END AS rfm_score,
    recency_days,
    monetary_value,
    n_orders
FROM rfm
ON CONFLICT (customer_unique_id) DO UPDATE
SET
    segment_name   = EXCLUDED.segment_name,
    rfm_score      = EXCLUDED.rfm_score,
    recency_days   = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders       = EXCLUDED.n_orders,
    assigned_at    = NOW();
"""

with engine.begin() as conn:
    conn.execute(text(customer_segments_insert_sql))

segments_check = pd.read_sql("""
SELECT
    segment_name,
    COUNT(*) AS customers,
    ROUND(AVG(monetary_value), 2) AS avg_monetary_value,
    ROUND(AVG(n_orders), 2) AS avg_orders
FROM customer_segments
GROUP BY segment_name
ORDER BY customers DESC;
""", engine)

segments_check

,segment_name,customers,avg_monetary_value,avg_orders
0,regular,58042,140.72,1.03
1,inactive,28427,142.20,1.03
2,new,9561,138.16,1.04
3,VIP,66,922.09,3.94


Найбільше покупців потрапило до сегмента regular - 58 042. Сегмент VIP найменший - 66 покупців, але він має найбільшу середню суму покупок та найбільшу середню кількість замовлень.

**Пояснення:** `customer_unique_id` використовую, тому що він показує одного реального покупця, а `customer_id` може бути різним для його замовлень. `olist_customer_dim` потрібна, щоб мати один рядок для кожного покупця та використовувати коректний foreign key. Для `reference_date` використовую максимальну дату з датасету, а не `CURRENT_DATE`, щоб результат не змінювався залежно від дати запуску notebook. `ON CONFLICT` дозволяє запускати сегментацію повторно: існуючі записи оновлюються, а дублікати не створюються.

## 4. JOIN-запити

### 4.1 INNER JOIN

**Business-question:** Які категорії товарів і штати продавців дають найбільший revenue у доставлених замовленнях?

Для цього об'єдную дані про товари, замовлення, категорії та продавців через INNER JOIN.

In [39]:
join_inner_sql = """
SELECT
    p.product_category_name,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
JOIN olist_products AS p
    ON p.product_id = oi.product_id
JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY
    p.product_category_name,
    s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10;
"""

inner_result = pd.read_sql(join_inner_sql, engine)
inner_result

,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


**Interpretation:** Найбільший revenue серед отриманих результатів має категорія `cama_mesa_banho` у штаті SP - 1 070 048.14 при 8 214 замовленнях. Також у топі знаходяться `relogios_presentes` та `beleza_saude`. Усі перші 10 результатів належать продавцям зі штату SP.

### 4.2 LEFT JOIN + COALESCE

**Business-question:** Які основні характеристики покупок можна отримати для кожного реального покупця?

Використовую LEFT JOIN, щоб не втратити покупців без пов'язаних замовлень або товарів, а COALESCE замінює відсутні числові значення на 0.

In [40]:
join_left_sql = """
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value) AS order_total
    FROM olist_customers AS c
    LEFT JOIN olist_orders AS o
        ON o.customer_id = c.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY o.order_id, c.customer_unique_id
)
SELECT
    d.customer_unique_id,
    d.customer_state,
    COALESCE(COUNT(ot.order_id), 0) AS n_orders,
    COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
    COALESCE(AVG(ot.order_total), 0)::NUMERIC(12, 2) AS avg_order_value
FROM olist_customer_dim AS d
LEFT JOIN order_totals AS ot
    ON ot.customer_unique_id = d.customer_unique_id
GROUP BY
    d.customer_unique_id,
    d.customer_state
ORDER BY total_spend DESC
LIMIT 10;
"""

left_result = pd.read_sql(join_left_sql, engine)
left_result

,customer_unique_id,customer_state,n_orders,total_spend,avg_order_value
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,13664.08
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,3785.82
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,7274.88
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,6929.31
4,459bef486812aa25204be022145caa62,ES,1,6922.21,6922.21
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,6726.66
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,6081.54
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,4809.44
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4764.34
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,4681.78


**Interpretation:** Найбільший total_spend серед отриманих результатів має покупець зі штату RJ - 13 664.08. Другий покупець також зі штату RJ має 2 замовлення із загальною сумою 7 571.63. LEFT JOIN дозволяє зберегти покупців навіть без пов'язаних даних, а COALESCE у такому випадку повертає 0 замість NULL.

### 4.3 FULL OUTER JOIN

**Business-question:** У яких штатах є покупці та продавці, а де представлена лише одна зі сторін?

Порівнюю кількість покупців і продавців у кожному штаті через FULL OUTER JOIN. Це дозволяє зберегти всі штати з обох наборів, навіть якщо відповідника з іншої сторони немає.

In [41]:
join_full_sql = """
WITH customers_by_state AS (
    SELECT
        customer_state AS state,
        COUNT(DISTINCT customer_unique_id) AS n_customers
    FROM olist_customers
    GROUP BY customer_state
),
sellers_by_state AS (
    SELECT
        seller_state AS state,
        COUNT(DISTINCT seller_id) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)
SELECT
    COALESCE(c.state, s.state) AS state,
    COALESCE(c.n_customers, 0) AS n_customers,
    COALESCE(s.n_sellers, 0) AS n_sellers
FROM customers_by_state AS c
FULL OUTER JOIN sellers_by_state AS s
    ON c.state = s.state
ORDER BY n_customers DESC, n_sellers DESC;
"""

full_result = pd.read_sql(join_full_sql, engine)
full_result

,state,n_customers,n_sellers
0,SP,40302,1849
1,RJ,12384,171
2,MG,11259,244
3,RS,5277,129
4,PR,4882,349
5,SC,3534,190
6,BA,3277,19
7,DF,2075,30
8,ES,1964,23
9,GO,1952,40


**Interpretation:** Найбільше покупців і продавців знаходиться у штаті SP - 40 302 покупці та 1 849 продавців. У штатах AL, TO, AP і RR є покупці, але немає продавців. FULL OUTER JOIN дозволяє побачити такі випадки, оскільки зберігає записи з обох наборів навіть без відповідника.

### 4.4 SELF JOIN

**Business-question:** Які продавці знаходяться в одному місті та штаті?

Використовую SELF JOIN, тобто з'єдную таблицю продавців саму із собою. Умова `s1.seller_id < s2.seller_id` потрібна, щоб не порівнювати продавця із самим собою та не отримувати однакові пари двічі.

In [42]:
join_self_sql = """
SELECT
    s1.seller_id AS seller_1,
    s2.seller_id AS seller_2,
    s1.seller_city,
    s1.seller_state
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_city = s2.seller_city
   AND s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
ORDER BY
    s1.seller_state,
    s1.seller_city,
    s1.seller_id,
    s2.seller_id
LIMIT 20;
"""

self_result = pd.read_sql(join_self_sql, engine)
self_result

,seller_1,seller_2,seller_city,seller_state
0,1444c08e64d55fb3c25f0f09c07ffcf2,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA
1,398cb257329ef7af7f1943a8974a3cbc,75d34ebb1bd0bd7dde40dd507b8169c3,salvador,BA
2,398cb257329ef7af7f1943a8974a3cbc,a3dd39f583bc80bd8c5901c95878921e,salvador,BA
3,398cb257329ef7af7f1943a8974a3cbc,b00af24704019bd2e1b335e70ad11f7c,salvador,BA
4,398cb257329ef7af7f1943a8974a3cbc,eb4a59a06b3948e851a7d7a830bcbc42,salvador,BA
5,398cb257329ef7af7f1943a8974a3cbc,f3055483058c84d75ca2a3314fb4deef,salvador,BA
6,398cb257329ef7af7f1943a8974a3cbc,fc59392d66ef99377e50356ee4f3b4e1,salvador,BA
7,75d34ebb1bd0bd7dde40dd507b8169c3,a3dd39f583bc80bd8c5901c95878921e,salvador,BA
8,75d34ebb1bd0bd7dde40dd507b8169c3,b00af24704019bd2e1b335e70ad11f7c,salvador,BA
9,75d34ebb1bd0bd7dde40dd507b8169c3,eb4a59a06b3948e851a7d7a830bcbc42,salvador,BA


**Interpretation:** SELF JOIN показав пари різних продавців, які знаходяться в одному місті та штаті. У перших результатах багато пар продавців із Salvador, BA. Умова `s1.seller_id < s2.seller_id` допомагає не отримувати однакові пари двічі та не порівнювати продавця із самим собою.

### 4.5 Anti-join

**Business-question:** Які товари жодного разу не зустрічаються у замовленнях?

Використовую LEFT JOIN ... IS NULL, щоб знайти товари, для яких немає відповідного запису в order_items.

In [43]:
join_anti_sql = """
SELECT
    p.product_id,
    p.product_category_name
FROM olist_products AS p
LEFT JOIN olist_order_items AS oi
    ON oi.product_id = p.product_id
WHERE oi.product_id IS NULL
ORDER BY p.product_id;
"""

anti_result = pd.read_sql(join_anti_sql, engine)

print("Products without orders:", len(anti_result))
anti_result.head(20)

Products without orders: 0


,product_id,product_category_name


**Interpretation:** Anti-join не знайшов товарів без замовлень. Отже, усі товари з таблиці `olist_products` мають відповідний запис у `olist_order_items`.

## 5. GROUP BY, HAVING та агрегати

### 5.1 Revenue за категоріями

**Business-question:** Які категорії товарів мають найбільший revenue серед категорій із достатньою кількістю замовлень?

Групую дані за категоріями товарів та використовую HAVING, щоб залишити тільки категорії, які мають не менше 100 замовлень.

In [44]:
group_category_sql = """
SELECT
    COALESCE(p.product_category_name, '(unknown)') AS category,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue,
    AVG(oi.price)::NUMERIC(12, 2) AS avg_price
FROM olist_order_items AS oi
LEFT JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY COALESCE(p.product_category_name, '(unknown)')
HAVING COUNT(DISTINCT oi.order_id) >= 100
ORDER BY revenue DESC
LIMIT 10;
"""

category_result = pd.read_sql(group_category_sql, engine)
category_result

,category,n_orders,revenue,avg_price
0,beleza_saude,8836,1441248.07,130.16
1,relogios_presentes,5624,1305541.61,201.14
2,cama_mesa_banho,9417,1241681.72,93.30
3,esporte_lazer,7720,1156656.48,114.34
4,informatica_acessorios,6689,1059272.40,116.51
5,moveis_decoracao,6449,902511.79,87.56
6,utilidades_domesticas,5884,778397.77,90.79
7,cool_stuff,3632,719329.95,167.36
8,automotivo,3897,685384.32,139.96
9,ferramentas_jardim,3518,584219.21,111.63


**Interpretation:** Найбільший revenue має категорія `beleza_saude` - 1 441 248.07 при 8 836 замовленнях. Найбільша кількість замовлень у `cama_mesa_banho` - 9 417. Це показує, що більша кількість замовлень не завжди означає найбільший revenue.

### 5.2 Revenue продавців

**Business-question:** Які продавці з достатньою кількістю замовлень мають найбільший revenue?

Групую дані за продавцями та використовую HAVING, щоб аналізувати лише продавців, які мають не менше 50 замовлень.

In [45]:
group_seller_sql = """
SELECT
    s.seller_id,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue,
    AVG(oi.price)::NUMERIC(12, 2) AS avg_item_price
FROM olist_sellers AS s
JOIN olist_order_items AS oi
    ON oi.seller_id = s.seller_id
GROUP BY
    s.seller_id,
    s.seller_state
HAVING COUNT(DISTINCT oi.order_id) >= 50
ORDER BY revenue DESC
LIMIT 10;
"""

seller_metric_result = pd.read_sql(group_seller_sql, engine)
seller_metric_result

,seller_id,seller_state,n_orders,revenue,avg_item_price
0,4869f7a5dfa277a7dca6462dcf3b52b2,SP,1132,249640.70,198.51
1,7c67e1448b00f6e969d365cea6b010ab,SP,982,239536.44,137.77
2,53243585a1d6dc2643021fd1853d8905,BA,358,235856.68,543.36
3,4a3ca9315b744ce9f8e9374361493884,SP,1806,235539.96,100.89
4,fa1c13f2614d7b5c4749cbc52fecda94,SP,585,204084.73,331.13
5,da8622b14eb17ae2831f4ac5b9dab84a,SP,1314,185192.32,103.31
6,7e93a43ef30c4f03f38b393420bc753a,SP,336,182754.05,518.92
7,1025f0e2d44d7041d6cf58b6550e0bfa,SP,915,172860.69,97.32
8,7a67c85e85bb2ce8582c35f2203ad736,SP,1160,162648.38,121.05
9,955fee9216a65b617aa5c0531780ce60,SP,1287,160602.68,90.17


**Interpretation:** Найбільший revenue серед отриманих продавців становить 249 640.70 при 1 132 замовленнях. Продавець із найбільшою кількістю замовлень у топі має 1 806 замовлень, але нижчий revenue - 235 539.96.Більшість продавців у топ-10 знаходяться у штаті SP, але є також продавець зі штату BA.

### 5.3 Категорії покупок через STRING_AGG

**Business-question:** Які категорії товарів купували покупці з найбільшими загальними витратами?

Використовую STRING_AGG, щоб зібрати всі унікальні категорії покупок кожного реального покупця в один рядок.

In [46]:
group_string_sql = """
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
    STRING_AGG(
        DISTINCT COALESCE(p.product_category_name, '(unknown)'),
        ', ' ORDER BY COALESCE(p.product_category_name, '(unknown)')
    ) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi
    ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 1
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;
"""

string_result = pd.read_sql(group_string_sql, engine)
string_result

,customer_unique_id,n_orders,total_spend,categories
0,0a0a92112bd4c708ca5fde585afaa872,1,13440.00,telefonia_fixa
1,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,eletroportateis
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7160.00,telefonia_fixa
3,dc4802a71eae9be1dd28f5d788ceb526,1,6735.00,utilidades_domesticas
4,459bef486812aa25204be022145caa62,1,6729.00,pcs
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6499.00,artes
6,4007669dec559734d6f53e029e360987,1,5934.60,agro_industria_e_comercio
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4690.00,eletroportateis
8,5d0a2980b292d049061542014e8960bf,1,4599.90,cool_stuff
9,48e1ac109decbb87765a3eade6854098,1,4590.00,pcs


**Interpretation:** Найбільший total_spend серед отриманих покупців становить 13 440.00 у категорії `telefonia_fixa`. STRING_AGG дозволяє зібрати унікальні категорії покупок кожного покупця в одному полі.

**WHERE vs HAVING:** `WHERE` фільтрує окремі рядки до виконання GROUP BY, а `HAVING` фільтрує вже сформовані групи після агрегації. У цьому запиті HAVING залишає покупців, які мають щонайменше одне замовлення.

## 6. Set operations, window function та Reflection

### 6.1 INTERSECT

**Business-question:** У яких штатах є одночасно і покупці, і продавці?

Використовую INTERSECT, оскільки він повертає тільки ті значення, які присутні в обох наборах даних.

In [47]:
intersect_sql = """
SELECT customer_state AS state
FROM olist_customers

INTERSECT

SELECT seller_state AS state
FROM olist_sellers

ORDER BY state;
"""

intersect_result = pd.read_sql(intersect_sql, engine)

print("States with customers and sellers:", len(intersect_result))
intersect_result

States with customers and sellers: 23


,state
0,AC
1,AM
2,BA
3,CE
4,DF
5,ES
6,GO
7,MA
8,MG
9,MS


**Interpretation:** INTERSECT показав 23 штати, які присутні одночасно серед покупців і продавців. Цей оператор підходить для такого запиту, оскільки залишає тільки спільні значення з двох наборів даних.

### 6.2 ROW_NUMBER()

**Business-question:** Яке замовлення є найновішим для кожного реального покупця?

Використовую ROW_NUMBER(), щоб пронумерувати замовлення кожного покупця від найновішого до найстарішого. Потім залишаю тільки рядки з rn = 1.

In [48]:
window_sql = """
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_purchase_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
        ) AS rn
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    customer_unique_id,
    customer_id,
    order_id,
    order_purchase_timestamp
FROM ranked_orders
WHERE rn = 1
ORDER BY customer_unique_id
LIMIT 20;
"""

window_result = pd.read_sql(window_sql, engine)
window_result

,customer_unique_id,customer_id,order_id,order_purchase_timestamp
0,0000366f3b9a7992bf8c76cfdf3221e2,fadbb3709178fc513abc1b2670aa1ad2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4cb282e167ae9234755102258dd52ee8,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,9b3932a6253894a02c1df9d19004239f,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,914991f0c02ef0843c0e7010c819d642,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,47227568b10f5f58a524a75507e6992c,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42
5,0004bd2a26a76fe21f786e4fbd80607f,4a913a170c26e3c8052ed0202849b5a8,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16
6,00050ab1314c0e55a6ca13cf7181fecf,d2509c13692836fc0449e88cf9eb4858,d0028facea13f508e880202d7097a5a1,2018-04-20 12:57:23
7,00053a61a98854899e70ed204dd4bafe,a81ebb9b32f102298c0c89635b4b3154,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41
8,0005e1862207bf6ccc02e4228effd9a0,3b37fb626fdf46cd99d37ec62afa88ff,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12
9,0005ef4cd20d2893f0d9fbd94d3c0d97,c59e8ff99836e90d8b457d4122dc34e9,01b330808c5819a6a3cb79b72f0b8288,2018-03-12 15:22:12


**Interpretation:** ROW_NUMBER() нумерує замовлення окремо для кожного `customer_unique_id`, починаючи з найновішого. Після фільтрації `rn = 1` у результаті залишається останнє замовлення кожного покупця.

## Reflection

У цій домашній роботі я працювала з PostgreSQL та датасетом Olist Brazilian E-commerce. Спочатку я завантажила дані та створила raw-таблиці. Потім на їх основі створила typed-таблиці, де вже вказала правильні типи даних, ключі та різні обмеження. Також перевірила кількість рядків, щоб переконатися, що під час перенесення даних нічого не втратилося.

Далі я працювала з DML-запитами. Використовувала INSERT, UPDATE та DELETE, а також RETURNING і ON CONFLICT. Мені було цікаво побачити, як за допомогою RETURNING можна одразу отримати результат зміни, а ON CONFLICT дозволяє повторно запускати запит без створення дублікатів.

Окремо я створила сегментацію покупців. Тут було важливо використовувати customer_unique_id, тому що один покупець може мати різні customer_id. Також я рахувала кількість замовлень, суму покупок та інші показники.

У наступній частині я зробила різні JOIN-запити: INNER JOIN, LEFT JOIN, FULL OUTER JOIN, SELF JOIN та anti-join. На цих прикладах мені стало зрозуміліше, чим вони відрізняються і коли який JOIN краще використовувати.

Також у роботі я використовувала GROUP BY, HAVING, STRING_AGG, INTERSECT та ROW_NUMBER(). Найскладніше для мене було правильно об'єднувати таблиці та слідкувати, щоб через повторення рядків не отримати неправильні результати. Після цього завдання я стала краще розуміти, як працювати з декількома пов'язаними таблицями та як перевіряти результати SQL-запитів.Також я навчилася краще розуміти структуру таблиць перед тим, як писати складніші запити.